In [ ]:
API_KEY = "YOUR_API_KEY"

In [ ]:
from inference_sdk import InferenceHTTPClient

CLIENT = InferenceHTTPClient(
    api_url="http://localhost:9001",
    api_key=API_KEY
)

image_url = "https://source.roboflow.com/rbYMofzvPcWgsZ193zII4ZKvVEb2/2MuDYNlv1trcH1acW5Kk/original.jpg"
model_id = "rfdetr-seg-preview"
result = CLIENT.infer(image_url, model_id=model_id)

In [ ]:
import requests
from io import BytesIO
import numpy as np
from PIL import Image
import supervision as sv

response = requests.get(image_url)
image = np.array(Image.open(BytesIO(response.content)))[:, :, ::-1] # BGR -> RGB

mask_annotator = sv.MaskAnnotator()

detections = sv.Detections.from_inference(result)
annotated = mask_annotator.annotate(scene=image.copy(), detections=detections)
sv.plot_image(annotated)

In [ ]:
from copy import deepcopy
# filter out ego-vehicle (simple heuristic: the widest mask - ego take up most of the image width)
ego_pred_idx = np.argmax([p["width"] for p in result["predictions"]])
result_without_ego = deepcopy(result)
result_without_ego["predictions"].pop(ego_pred_idx)

detections_without_ego = sv.Detections.from_inference(result_without_ego)
annotated_without_ego = mask_annotator.annotate(scene=image.copy(), detections=detections_without_ego)
sv.plot_image(annotated_without_ego)

In [ ]:
import json
from itertools import chain

mask_input = [list(chain.from_iterable([ [pt["x"],pt["y"]] for pt in p["points"] ])) for p in result_without_ego["predictions"]]

sam3_3d_result = CLIENT.sam3_3d_infer(inference_input=image_url, mask_input=mask_input)

In [ ]:
from base64 import b64decode

# 1. Scene Mesh (GLB)
scene_mesh_glb = sam3_3d_result["mesh_glb"]
if scene_mesh_glb is not None:
    scene_mesh_glb = b64decode(scene_mesh_glb)
    print(f"\n[Output 1/3] Scene Mesh (GLB format)")
    with open("out_mesh.glb", "wb") as f:
        f.write(scene_mesh_glb)
    print(f"  Saved mesh to out_mesh.glb ({len(scene_mesh_glb):,} bytes)")
else:
    print(f"\n[Output 1/3] No mesh output")

# 2. Combined Gaussian splatting
scene_gaussian_ply = sam3_3d_result["gaussian_ply"]
if scene_gaussian_ply is not None:
    scene_gaussian_ply = b64decode(scene_gaussian_ply)
    print(f"\n[Output 2/3] Combined Gaussian splatting (PLY format)")
    with open("out_gaussian.ply", "wb") as f:
        f.write(scene_gaussian_ply)
    print(f"  Saved gaussian to out_gaussian.ply ({len(scene_gaussian_ply):,} bytes)")
else:
    print(f"\n[Output 2/3] No combined gaussian output")

# 3. Individual objects
objects = sam3_3d_result["objects"]
print(f"\n[Output 3/3] Individual objects ({len(objects)} objects)")
objects_metadata = []
for i, obj in enumerate(objects):
    print(f"\n  Object {i}:")

    # Save individual mesh
    obj_mesh_glb = obj["mesh_glb"]
    if obj_mesh_glb is not None:
        obj_mesh_glb = b64decode(obj_mesh_glb)
        filename = f"out_object_{i}_mesh.glb"
        with open(filename, "wb") as f:
            f.write(obj_mesh_glb)
        print(f"    Saved mesh to {filename} ({len(obj_mesh_glb):,} bytes)")

    # Save individual gaussian
    obj_gaussian_ply = obj["gaussian_ply"]
    if obj_gaussian_ply is not None:
        obj_gaussian_ply = b64decode(obj_gaussian_ply)
        filename = f"out_object_{i}_gaussian.ply"
        with open(filename, "wb") as f:
            f.write(obj_gaussian_ply)
        print(f"    Saved gaussian to {filename} ({len(obj_gaussian_ply):,} bytes)")

    # Collect metadata
    obj_metadata = deepcopy(obj["metadata"])
    obj_metadata["object_index"] = i
    objects_metadata.append(obj_metadata)
    print(f'    Metadata: rotation={obj["metadata"]["rotation"] is not None}, translation={obj["metadata"]["translation"] is not None}, scale={obj["metadata"]["scale"] is not None}')

# Save all metadata to file
with open("out_metadata.json", "w") as f:
    json.dump({"objects": objects_metadata}, f, indent=2)
print(f"\n  Saved all metadata to out_metadata.json")

print("\n" + "=" * 80)
print("All outputs saved successfully!")
print("=" * 80)

In [ ]:
import rerun as rr

rr.init("rf-mot3d", spawn=True)

rr.log("object_0/mesh", rr.Asset3D(path="out_object_0_mesh.glb"))
rr.notebook_show()
